# Hybrid Quantum GNNs for Molecular Toxicity Classification
**BMS Institute of Technology and Management** | ML – BCS602 | AY 2025–2026  
**Guide:** Dr. Nagabhushan SV

| USN | Name |
|---|---|
| 1BY23CS014 | Aishwarya J A |
| 1BY23CS026 | Anurag Rai |
| 1BY23CS053 | Dasiga Venkata Ashish Kumar |
| 1BY23CS068 | G Nithish |

## 1. Install Dependencies

In [ ]:
!pip install torch torch-geometric rdkit pennylane scikit-learn pandas tqdm matplotlib seaborn requests -q

## 2. Clone Repository (Colab only)

In [ ]:
try:
    import google.colab
    # Remove existing repo if it exists (for notebook re-runs in Colab)
    try:
        !rm -rf /content/HQGNN
    except:
        pass
    !git clone https://github.com/r-anurag/Hybrid-Quantum-GNNs-for-Molecular-Toxicity-Classification.git /content/HQGNN
    %cd /content/HQGNN
    !ls -la
    print('Repository cloned successfully')
except:
    print('Running locally, skipping clone')

## 3. Setup Environment

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings("ignore")

# Detect environment
try:
    import google.colab
    IN_COLAB = True
except:
    IN_COLAB = False

# Change to src directory
if os.path.exists('src'):
    os.chdir('src')
elif os.path.exists('../src'):
    os.chdir('../src')
else:
    print('ERROR: src/ folder not found')
    print(f'Current directory: {os.getcwd()}')
    print(f'Contents: {os.listdir()}')

print(f"Environment: {'Colab' if IN_COLAB else 'Local'}")
print(f"Working directory: {os.getcwd()}")

## 4. Import Libraries

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

# Set device
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch: {torch.__version__}")
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 5. Setup Checkpoint Directory

In [ ]:
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        CHECKPOINT_DIR = '/content/drive/MyDrive/HQGNN_checkpoints'
        print(f"Checkpoints will be saved to: {CHECKPOINT_DIR}")
    except:
        CHECKPOINT_DIR = None
        print("No checkpoint directory (Drive not mounted)")
else:
    CHECKPOINT_DIR = '../checkpoints'
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    print(f"Checkpoints will be saved to: {CHECKPOINT_DIR}")

## 6. Load Datasets

In [ ]:
from data_pipeline import load_dataset

print("Loading datasets...\n")
datasets = {}

for name in ("clintox", "tox21"):
    print(f"Loading {name.upper()}...")
    data_list, class_weights, tasks = load_dataset(name)
    datasets[name] = (data_list, class_weights, tasks)
    print(f"  {len(data_list)} molecules, {len(tasks)} tasks")
    print(f"  Node features: {data_list[0].x.shape[1]}")
    print(f"  Edge features: {data_list[0].edge_attr.shape[1]}\n")

## 7. Define Model Variants

In [ ]:
from models import GCN, HybridQGNN, QuantumOnly

# Configuration
IN_CHANNELS = 10
EPOCHS = 50
LR = 1e-3
BATCH = 64
N_FOLDS = 5

def get_variants(num_tasks):
    return {
        "Classical GCN": lambda: GCN(
            IN_CHANNELS, hidden=64, embed_dim=32, num_tasks=num_tasks
        ),
        "Quantum-only 4-qubit": lambda: QuantumOnly(
            IN_CHANNELS, n_qubits=4, n_layers=2, num_tasks=num_tasks
        ),
        "Hybrid 4-qubit": lambda: HybridQGNN(
            IN_CHANNELS, gcn_hidden=64, gcn_embed=32,
            n_qubits=4, n_layers=2, num_tasks=num_tasks
        ),
        "Hybrid 8-qubit": lambda: HybridQGNN(
            IN_CHANNELS, gcn_hidden=64, gcn_embed=32,
            n_qubits=8, n_layers=2, num_tasks=num_tasks
        ),
        "Hybrid 4-qubit + Edge": lambda: HybridQGNN(
            IN_CHANNELS, gcn_hidden=64, gcn_embed=32,
            n_qubits=4, n_layers=2, num_tasks=num_tasks, edge_embed=True
        ),
    }

print(f"Configuration:")
print(f"  Epochs: {EPOCHS}")
print(f"  Learning rate: {LR}")
print(f"  Batch size: {BATCH}")
print(f"  Cross-validation folds: {N_FOLDS}")
print(f"  Device: {DEVICE}")

## 8. Run Experiments

In [ ]:
from evaluate import cross_validate
import time

all_rows = []
start_time = time.time()

for ds_name, (data_list, class_weights, tasks) in datasets.items():
    num_tasks = len(tasks)
    print(f"\n{'='*70}")
    print(f"Dataset: {ds_name.upper()}  ({len(data_list)} molecules, {num_tasks} tasks)")
    print(f"{'='*70}")

    for name, model_fn in get_variants(num_tasks).items():
        print(f"\n>> {name}")
        
        model_temp = model_fn()
        n_params = sum(p.numel() for p in model_temp.parameters() if p.requires_grad)
        del model_temp
        print(f"   Parameters: {n_params:,}")

        results = cross_validate(
            model_fn, data_list, class_weights,
            n_splits=N_FOLDS, epochs=EPOCHS, lr=LR,
            batch_size=BATCH, device=DEVICE, verbose=False,
            checkpoint_dir=CHECKPOINT_DIR,
            model_name=name.replace(" ", "_"),
            dataset_name=ds_name
        )
        
        row = {
            "Model": name,
            "Dataset": ds_name,
            "Params": n_params,
            "auc_mean": results['auc_mean'],
            "auc_std": results['auc_std'],
            "f1_mean": results['f1_mean'],
            "f1_std": results['f1_std'],
            "time_mean": results['time_mean'],
            "histories": results['histories']
        }
        all_rows.append(row)
        
        print(f"   AUC: {results['auc_mean']:.4f} ± {results['auc_std']:.4f}")
        print(f"   F1:  {results['f1_mean']:.4f} ± {results['f1_std']:.4f}")
        print(f"   Time/epoch: {results['time_mean']:.2f}s")

elapsed = time.time() - start_time
print(f"\n{'='*70}")
print(f"Total time: {elapsed/60:.1f} minutes")
print(f"{'='*70}")

df = pd.DataFrame([{k: v for k, v in r.items() if k != "histories"} for r in all_rows])
os.makedirs("../results", exist_ok=True)
df.to_csv("../results/results.csv", index=False)
print("\nResults saved to ../results/results.csv")

## 9. Results Table

In [ ]:
display_df = df[[
    "Dataset", "Model", "Params",
    "auc_mean", "auc_std",
    "f1_mean", "f1_std",
    "time_mean"
]].round(4)

print("\nExperimental Results:")
print("="*100)
print(display_df.to_string(index=False))
print("="*100)

## 10. Performance Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, metric, label in zip(axes, ["auc_mean", "f1_mean"], ["ROC-AUC", "F1-Score"]):
    for i, ds_name in enumerate(["clintox", "tox21"]):
        sub = df[df["Dataset"] == ds_name]
        x = np.arange(len(sub)) + i * 0.35
        ax.bar(
            x, sub[metric],
            width=0.35,
            yerr=sub[metric.replace("mean", "std")],
            capsize=4,
            label=ds_name.upper(),
            alpha=0.8
        )
    
    ax.set_xticks(np.arange(len(sub)) + 0.175)
    ax.set_xticklabels(sub["Model"], rotation=45, ha='right')
    ax.set_title(f"{label} Comparison", fontsize=14, fontweight='bold')
    ax.set_ylabel(label, fontsize=12)
    ax.set_ylim(0, 1)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("../results/performance_comparison.png", dpi=150, bbox_inches='tight')
plt.show()

## 11. Training Curves

In [ ]:
n_plots = min(len(all_rows), 10)
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for idx, row in enumerate(all_rows[:n_plots]):
    ax = axes[idx]
    histories = row["histories"]
    
    train_losses = np.mean([h["train_loss"] for h in histories], axis=0)
    val_losses = np.mean([h["val_loss"] for h in histories], axis=0)
    
    ax.plot(train_losses, label="Train", linewidth=2, alpha=0.8)
    ax.plot(val_losses, label="Val", linewidth=2, alpha=0.8)
    ax.set_title(f"{row['Model']}\n{row['Dataset'].upper()}", fontsize=10)
    ax.set_xlabel("Epoch", fontsize=9)
    ax.set_ylabel("Loss", fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

for idx in range(n_plots, len(axes)):
    axes[idx].axis('off')

plt.tight_layout()
plt.savefig("../results/training_curves.png", dpi=150, bbox_inches='tight')
plt.show()